# Voice Explorer (Jupyter POC)

Interactive sound library browser with audio playback.

In [ ]:
import json
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Audio, clear_output
from pathlib import Path

LIB = Path.home() / '.claude/local/voice/sound_library.json'
df = pd.DataFrame(json.loads(LIB.read_text()))
print(f'{len(df)} sounds loaded across {df["theme"].nunique()} themes')

In [ ]:
# Voice State
STATE = Path.home() / '.claude/local/voice/speaking-now.json'
try:
    data = json.loads(STATE.read_text().strip())
    print(f"Speaking: {data.get('speaking_pane', 'none')} | STT: {'active' if data.get('stt_active') else 'idle'} | Muted: {data.get('muted', False)}")
except:
    print('State unavailable')

In [ ]:
# Filters
theme_dd = widgets.Dropdown(options=['all'] + sorted(df['theme'].unique().tolist()), value='all', description='Theme:')
cat_dd = widgets.Dropdown(options=['all'] + sorted(df['slot'].unique().tolist()), value='all', description='Category:')
player_output = widgets.Output()
table_output = widgets.Output()

def update(change=None):
    filtered = df.copy()
    if theme_dd.value != 'all':
        filtered = filtered[filtered['theme'] == theme_dd.value]
    if cat_dd.value != 'all':
        filtered = filtered[filtered['slot'] == cat_dd.value]
    with table_output:
        clear_output(wait=True)
        display(filtered[['theme', 'slot', 'variant', 'name', 'duration_ms', 'peak_db', 'rms_db']].reset_index(drop=True))
    return filtered

theme_dd.observe(update, names='value')
cat_dd.observe(update, names='value')

display(widgets.HBox([theme_dd, cat_dd]))
display(table_output)
display(player_output)
update()

In [ ]:
# Click-to-play: enter a row number to play that sound
row_input = widgets.IntText(value=0, description='Row #:', min=0)
play_btn = widgets.Button(description='Play', button_style='primary')

def on_play(btn):
    filtered = df.copy()
    if theme_dd.value != 'all':
        filtered = filtered[filtered['theme'] == theme_dd.value]
    if cat_dd.value != 'all':
        filtered = filtered[filtered['slot'] == cat_dd.value]
    filtered = filtered.reset_index(drop=True)
    idx = row_input.value
    if idx < len(filtered):
        row = filtered.iloc[idx]
        with player_output:
            clear_output(wait=True)
            print(f"Playing: {row['theme']} / {row['name']} | {row['duration_ms']}ms | {row['rms_db']}dB RMS")
            display(Audio(filename=row['path'], autoplay=True))

play_btn.on_click(on_play)
display(widgets.HBox([row_input, play_btn]))

In [ ]:
# Library Stats
print(f"Total: {len(df)} sounds, {df['theme'].nunique()} themes, {df['slot'].nunique()} slots")
print(f"Total duration: {df['duration_ms'].sum() / 1000:.0f}s")
print(f"Avg RMS: {df['rms_db'].mean():.1f} dBFS | Avg Peak: {df['peak_db'].mean():.1f} dBFS")
print()
df.groupby('theme').agg(count=('name', 'count'), avg_rms=('rms_db', 'mean')).round(1)